<a href="https://colab.research.google.com/github/susara20010420/Workplace-Safety-Insights-from-the-Industrial-Safety-Health-Analytics-Dataset/blob/main/WHSAT_text_transformer_updatedbyBYS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets scikit-learn torch

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/susara20010420/Workplace-Safety-Insights-from-the-Industrial-Safety-Health-Analytics-Dataset/main/cleaned_safety_data_set_B.csv")

# Keep only necessary columns
data_set = df[['description', 'severity', 'potential_severity', 'critical_risk', 'near_miss_gap']].dropna()
data_set.head()

,description,severity,potential_severity,critical_risk,near_miss_gap
0,While removing the drill rod of the Jumbo 08 f...,1,4,Pressed,3
1,During the activation of a sodium sulphide pum...,1,4,Pressurized Systems,3
2,In the sub-station MILPO located at level +170...,1,3,Manual Tools,2
3,Being 9:45 am. approximately in the Nv. 1880 C...,1,1,Others,0
4,Approximately at 11:45 a.m. in circumstances t...,4,4,Others,0


In [3]:
#creating a dictionary with text label into numerical value. Eg: "Bees"==1
data_set['critical_risk_label'] = data_set['critical_risk'].astype('category').cat.codes
risk_labels = dict(enumerate(data_set['critical_risk'].astype('category').cat.categories))
print(risk_labels)

{0: '\nNot applicable', 1: 'Bees', 2: 'Blocking and isolation of energies', 3: 'Burn', 4: 'Chemical substances', 5: 'Confined space', 6: 'Cut', 7: 'Electrical Shock', 8: 'Electrical installation', 9: 'Fall', 10: 'Fall prevention', 11: 'Fall prevention (same level)', 12: 'Individual protection equipment', 13: 'Liquid Metal', 14: 'Machine Protection', 15: 'Manual Tools', 16: 'Others', 17: 'Plates', 18: 'Poll', 19: 'Power lock', 20: 'Pressed', 21: 'Pressurized Systems', 22: 'Pressurized Systems / Chemical Substances', 23: 'Projection', 24: 'Projection of fragments', 25: 'Projection/Burning', 26: 'Projection/Choco', 27: 'Projection/Manual Tools', 28: 'Suspended Loads', 29: 'Traffic', 30: 'Vehicles and Mobile Equipment', 31: 'Venomous Animals', 32: 'remains of choco'}


In [4]:
#to make 0–4
#needed for classification models which expect 0-indexed labels
data_set['severity_label'] = data_set['severity'] - 1
data_set['severity_label'].value_counts()

,count
severity_label,
0,316
1,40
2,31
3,30
4,8


In [5]:
#splitting the data set as trainning category (80%) and testing category (20%)
#ensures severity distribution is balanced in both sets (important for imbalanced data)
train_df, test_df = train_test_split(data_set, test_size=0.2, stratify=data_set['severity_label'], random_state=42)

In [ ]:
#tokenizing the description into IDs
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=False,
    )


Pay more attention from here onwards

In [7]:
from torch.utils.data import Dataset

# Converts everything to PyTorch tensors
class SafetyDataset(Dataset):
    def __init__(self, texts, labels, severity, potential_severity, near_miss_gap):
        self.encodings = tokenize(texts)  # This converts all your text descriptions into numbers that the model can understand
        self.labels = list(labels)
        self.severity = list(severity)
        self.potential_severity = list(potential_severity)
        self.near_miss_gap = list(near_miss_gap)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        item['severity'] = torch.tensor(self.severity[idx])
        item['potential_severity'] = torch.tensor(self.potential_severity[idx])
        item['near_miss_gap'] = torch.tensor(self.near_miss_gap[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [8]:
train_dataset = SafetyDataset(train_df['description'], train_df['critical_risk_label'], train_df['severity'], train_df['potential_severity'], train_df['near_miss_gap'])
test_dataset = SafetyDataset(test_df['description'], test_df['critical_risk_label'], test_df['severity'], test_df['potential_severity'], test_df['near_miss_gap'])

num_labels = data_set['critical_risk_label'].nunique()
print(num_labels)

33


In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', # treats "Hello" and "hello" as the same word
    num_labels=num_labels
)

In [10]:
from transformers import Trainer, TrainingArguments
import os
os.environ['TENSORBOARD_LOGGING_DIR'] = './logs'  # ✅ NEW WAY

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="no"
)

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "f1": f1_score(labels, preds, average='macro')
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [13]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

save_dir = "/content/drive/MyDrive/WHSAT/"
os.makedirs(save_dir, exist_ok=True)

# Get predictions from the trained model
predictions = trainer.predict(test_dataset)
preds = predictions.predictions
y_test = predictions.label_ids

np.save(os.path.join(save_dir, "transformer_probs.npy"), preds)
np.save(os.path.join(save_dir, "y_test.npy"), y_test)
np.save(os.path.join(save_dir, "test_indices.npy"), test_df.index.values)

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
from transformers import DistilBertModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load the base DistilBERT model (without classification head)
bert_extractor = DistilBertModel.from_pretrained('distilbert-base-uncased')
bert_extractor.to(device)
bert_extractor.eval()

def extract_cls_embeddings(texts, batch_size=64):
    """Return a numpy array of shape (len(texts), 768)."""
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors='pt'
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            outputs = bert_extractor(**encoded)
        # CLS token is the first one (index 0)
        cls_embs = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embs.append(cls_embs)
    return np.vstack(all_embs)

# Extract for the entire dataset (or only for train/test if you prefer)
# Here we use the original 'data_set' (before splitting)
all_texts = data_set['description'].tolist()
embeddings = extract_cls_embeddings(all_texts)

# Save as .npy
np.save(os.path.join(save_dir, "distilbert_embeddings.npy"), embeddings)

print(f"Saved embeddings shape: {embeddings.shape}")   # (N, 768)


# ===============================
# 2. Save prediction probabilities as CSV
# ===============================
# We already have predictions from the trainer on the test set.
# If you want probabilities for the *test* set only:
if 'predictions' in locals():  # if you already ran trainer.predict()
    # predictions.predictions are logits; convert to probabilities
    probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=-1).numpy()
    # Build a DataFrame with class probabilities
    class_names = [f"class_{i}" for i in range(num_labels)]
    prob_df = pd.DataFrame(probs, columns=class_names)
    # Add original test indices and true labels if you like
    prob_df['index'] = test_df.index.values
    prob_df['true_label'] = test_df['critical_risk_label'].values
    # Save as CSV
    prob_df.to_csv(os.path.join(save_dir, "distilbert_outputs.csv"), index=False)
    print("Saved prediction probabilities (test set) to distilbert_outputs.csv")
else:
    # If you haven't run trainer.predict() yet, run it now:
    predictions = trainer.predict(test_dataset)
    probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=-1).numpy()
    class_names = [f"class_{i}" for i in range(num_labels)]
    prob_df = pd.DataFrame(probs, columns=class_names)
    prob_df['index'] = test_df.index.values
    prob_df['true_label'] = test_df['critical_risk_label'].values
    prob_df.to_csv(os.path.join(save_dir, "distilbert_outputs.csv"), index=False)
    print("Saved prediction probabilities (test set) to distilbert_outputs.csv")

In [18]:
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import numpy as np

# Convert predictions (logits) to predicted labels
predicted_labels = np.argmax(preds, axis=1)

# Get the unique labels present in the true test labels
unique_test_labels = np.unique(y_test)

# Create target names only for the labels present in the test set
# Assuming 'risk_labels' dictionary is available from previous cells
# If 'risk_labels' is not directly available, you might need to recreate it or ensure it's in scope.
# For this fix, I'll use the 'risk_labels' variable as it's in the kernel state.
filtered_target_names = [risk_labels[label_idx] for label_idx in unique_test_labels]

# Calculate Confusion Matrix
cm = confusion_matrix(y_test, predicted_labels, labels=unique_test_labels)
print("Confusion Matrix:")
print(cm)

# Generate Classification Report (includes precision, recall, f1-score for each class)
print("\nClassification Report:")
print(classification_report(y_test, predicted_labels, labels=unique_test_labels, target_names=filtered_target_names, zero_division=0))

# Calculate Macro F1 Score
macro_f1 = f1_score(y_test, predicted_labels, average='macro')
print(f"\nMacro F1 Score: {macro_f1:.4f}")

# Calculate Micro F1 Score
micro_f1 = f1_score(y_test, predicted_labels, average='micro')
print(f"Micro F1 Score: {micro_f1:.4f}")

Confusion Matrix:
[[ 0  0  0  0  0  0  0  0  0  2  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  7  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  7  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  3  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  3  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  5  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0 34  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  4  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  3  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0

In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load the base DistilBERT model (without classification head)
bert_extractor = DistilBertModel.from_pretrained('distilbert-base-uncased')
bert_extractor.to(device)
bert_extractor.eval()

def extract_cls_embeddings(texts, batch_size=64):
    """Return a numpy array of shape (len(texts), 768)."""
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors='pt'
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            outputs = bert_extractor(**encoded)
        cls_embs = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embs.append(cls_embs)
    return np.vstack(all_embs)

# Extract for training and test sets
X_train_emb = extract_cls_embeddings(train_df['description'].tolist())
X_test_emb  = extract_cls_embeddings(test_df['description'].tolist())

# Save them (along with labels and indices) to your Drive folder
save_dir = "/content/drive/MyDrive/WHSAT/"
os.makedirs(save_dir, exist_ok=True)

np.save(os.path.join(save_dir, "X_train_emb.npy"), X_train_emb)
np.save(os.path.join(save_dir, "X_test_emb.npy"), X_test_emb)
np.save(os.path.join(save_dir, "y_train.npy"), train_df['severity_label'].values)
np.save(os.path.join(save_dir, "y_test.npy"), test_df['severity_label'].values)
np.save(os.path.join(save_dir, "test_indices.npy"), test_df.index.values)

print(f"Saved X_train_emb shape: {X_train_emb.shape}")
print(f"Saved X_test_emb shape: {X_test_emb.shape}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved X_train_emb shape: (340, 768)
Saved X_test_emb shape: (85, 768)
